<a href="https://colab.research.google.com/github/whgusdn5221/comfycolab/blob/main/comfy_real.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 0단계: 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
#@title 📥 1단계: 모델 풀 패키지 다운로드 (안전 구역)
#@markdown 원하는 모델을 체크하세요. `/content/storage/models`에 저장되어 본체 재설치 시에도 유지됩니다.

#@markdown ---
#@markdown ### **[1] Qwen 2.5 VL 전용 (최신 모델)**
qwen_25_vl_7b = True #@param {type:"boolean"}
#@markdown - qwen_2.5_vl_7b_fp8_scaled.safetensors (3584 규격 에러 해결용)

#@markdown ### **[2] 기존 Qwen & FLUX 메인**
qwen_edit_fp8 = True #@param {type:"boolean"}
flux_schnell_fp8 = True #@param {type:"boolean"}

#@markdown ### **[3] 텍스트 인코더 및 CLIP**
clip_flux_set = True #@param {type:"boolean"}
#@markdown - clip_l, t5xxl_fp8 포함

#@markdown ### **[4] 제어 및 특수 모델 (IP-Adapter, ControlNet, SCHP)**
ipadapter_sdxl = True #@param {type:"boolean"}
controlnet_pose = True #@param {type:"boolean"}
human_parsing_schp = True #@param {type:"boolean"}
#@markdown ---

import os
SAFE_BASE = "/content/storage/models"
folders = ["checkpoints/qwen", "clip", "vae", "loras", "pulid", "ipadapter", "clip_vision", "controlnet", "photomaker"]
for f in folders: os.makedirs(f"{SAFE_BASE}/{f}", exist_ok=True)

def dl(url, path, name):
    target = os.path.join(path, name)
    if not os.path.exists(target):
        print(f"📥 다운로드 중: {name}...")
        !wget -c "{url}" -O "{target}"
    else:
        print(f"✅ 보존됨: {name}")

# --- 실행부 ---

# [1] Qwen 2.5 VL (사용자 추가 요청분)
if qwen_25_vl_7b:
    # 이 모델은 노드에서 CLIP으로 인식하므로 clip 폴더에 저장합니다.
    dl("https://huggingface.co/Comfy-Org/Qwen2.5-VL-7B-Instruct-ComfyUI/resolve/main/qwen_2.5_vl_7b_fp8_scaled.safetensors", f"{SAFE_BASE}/clip", "qwen_2.5_vl_7b_fp8_scaled.safetensors")

# [2] 기존 Qwen & Flux
if qwen_edit_fp8:
    dl("https://huggingface.co/Comfy-Org/Qwen-Image-Edit_ComfyUI/resolve/main/qwen_image_edit_2511_fp8_e4m3fn.safetensors", f"{SAFE_BASE}/checkpoints/qwen", "qwen_image_edit_2511_fp8_e4m3fn.safetensors")
    dl("https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors", f"{SAFE_BASE}/vae", "qwen_image_vae.safetensors")

if flux_schnell_fp8:
    dl("https://huggingface.co/Comfy-Org/flux1-schnell/resolve/main/flux1-schnell-fp8.safetensors", f"{SAFE_BASE}/checkpoints", "flux1-schnell-fp8.safetensors")
    dl("https://huggingface.co/Comfy-Org/flux1-schnell/resolve/main/ae.safetensors", f"{SAFE_BASE}/vae", "ae.safetensors")

# [3] CLIP
if clip_flux_set:
    dl("https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors", f"{SAFE_BASE}/clip", "clip_l.safetensors")
    dl("https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors", f"{SAFE_BASE}/clip", "t5xxl_fp8_e4m3fn.safetensors")

# [4] 제어 모델들
if ipadapter_sdxl:
    dl("https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors", f"{SAFE_BASE}/clip_vision", "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors")
    dl("https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors", f"{SAFE_BASE}/ipadapter", "ip-adapter-plus_sdxl_vit-h.safetensors")

if controlnet_pose:
    dl("https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/thibaud_xl_openpose.safetensors", f"{SAFE_BASE}/controlnet", "OpenPoseXL2.safetensors")

if human_parsing_schp:
    dl("https://huggingface.co/ZhengGuo/MagicClothing/resolve/main/exp-schp-201908261155-lip.pth", f"{SAFE_BASE}/photomaker", "exp-schp-201908261155-lip.pth")

print("\n✨ 모델 준비 완료!")

✅ 보존됨: qwen_2.5_vl_7b_fp8_scaled.safetensors
✅ 보존됨: qwen_image_edit_2511_fp8_e4m3fn.safetensors
✅ 보존됨: qwen_image_vae.safetensors
📥 다운로드 중: flux1-schnell-fp8.safetensors...
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
--2026-03-28 17:58:39--  https://huggingface.co/Comfy-Org/flux1-schnell/resolve/main/flux1-schnell-fp8.safetensors
Resolving huggingface.co (huggingface.co)... 3.167.112.96, 3.167.112.45, 3.167.112.38, ...
Connecting to huggingface.co (huggingface.co)|3.167.112.96|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/66afcaacb9803b78f0616be8/efd9854e68978e1ec622df62472d68165837d7257ffaa5dce36d2f569ac28796?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Content-Sha256=UNSIGNED-PAYLOAD&X-Amz-Credential=cas%2F20260328%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260328T175839Z&X-Amz-Expires=3600&X-Amz-Signature=9ab4fd7faa1945c5ec1f76469b7aedde7

In [9]:
#@title 📦 2단계: 커스텀 노드 선택 설치 (안전 구역)
#@markdown 설치할 노드를 체크하세요. (안전 구역에 저장되어 본체 재설치 시에도 유지됩니다.)

#@markdown ---
node_manager = True #@param {type:"boolean"}
node_qwen_official = True #@param {type:"boolean"}
#@markdown - Qwen 공식 노드 (Comfy-Org)
node_qwen_adv = True #@param {type:"boolean"}
#@markdown - Qwen 고급 노드 (ZHO-ZHO-ZHO / _adv 버전)
node_ipadapter = True #@param {type:"boolean"}
node_controlnet = True #@param {type:"boolean"}
node_essentials = True #@param {type:"boolean"}
#@markdown ---

import os
SAFE_NODE_BASE = "/content/storage/custom_nodes"
os.makedirs(SAFE_NODE_BASE, exist_ok=True)
%cd {SAFE_NODE_BASE}

def git_dl(url, name):
    if not os.path.exists(name):
        print(f"📦 설치 중: {name}...")
        !git clone {url} {name}
    else: print(f"✅ 보존됨: {name}")

if node_manager: git_dl("https://github.com/ltdrdata/ComfyUI-Manager", "ComfyUI-Manager")
if node_qwen_official: git_dl("https://github.com/Comfy-Org/ComfyUI-Qwen-Image-Edit", "ComfyUI-Qwen-Image-Edit")
if node_qwen_adv: git_dl("https://github.com/ZHO-ZHO-ZHO/ComfyUI-Qwen-Image-Edit_adv", "ComfyUI-Qwen-Image-Edit_adv")
if node_ipadapter: git_dl("https://github.com/cubiq/ComfyUI_IPAdapter_plus", "ComfyUI_IPAdapter_plus")
if node_controlnet: git_dl("https://github.com/Fannovel16/comfyui_controlnet_aux", "comfyui_controlnet_aux")
if node_essentials: git_dl("https://github.com/cubiq/ComfyUI_essentials", "ComfyUI_essentials")

# Qwen 가동을 위한 필수 라이브러리 (이걸 안 깔면 노드가 안 뜹니다)
print("\n🛠️ Qwen 전용 필수 라이브러리 설치 중...")
!pip install transformers_stream_generator einops insightface onnxruntime-gpu blake3 -q

print("\n🚀 노드 준비 완료!")

/content/storage/custom_nodes
✅ 보존됨: ComfyUI-Manager
📦 설치 중: ComfyUI-Qwen-Image-Edit...
Cloning into 'ComfyUI-Qwen-Image-Edit'...
fatal: could not read Username for 'https://github.com': No such device or address
📦 설치 중: ComfyUI-Qwen-Image-Edit_adv...
Cloning into 'ComfyUI-Qwen-Image-Edit_adv'...
fatal: could not read Username for 'https://github.com': No such device or address
✅ 보존됨: ComfyUI_IPAdapter_plus
✅ 보존됨: comfyui_controlnet_aux
✅ 보존됨: ComfyUI_essentials

🛠️ Qwen 전용 필수 라이브러리 설치 중...

🚀 노드 준비 완료!


In [ ]:
#@title ⚡ 3단계: 소스 선택 및 메인 실행
#@markdown ### 📂 저장소 소스 선택 (모델/노드 위치)
source_selection = "Local_SSD" #@param ["Google_Drive", "Local_SSD"]

import os, subprocess, time, re
from google.colab import drive

LOCAL_PATH = "/content/ComfyUI"
SAFE_STORAGE = "/content/storage"
DRIVE_BASE = "/content/drive/MyDrive/ComfyUI"

# 1. 드라이브 마운트 (드라이브 선택 시 필수)
if source_selection == "Google_Drive":
    if not os.path.exists('/content/drive'):
        print("📂 구글 드라이브 마운트 중...")
        drive.mount('/content/drive')
    # 드라이브 내 폴더 구조 생성
    os.makedirs(f"{DRIVE_BASE}/models", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/custom_nodes", exist_ok=True)

# 2. 본체(코드) 경로 체크 및 복구
if not os.path.exists(LOCAL_PATH):
    print("⚠️ ComfyUI 본체가 없습니다. 새로 설치합니다...")
    %cd /content
    !git clone https://github.com/comfyanonymous/ComfyUI {LOCAL_PATH}
    !pip install -r {LOCAL_PATH}/requirements.txt -q
else:
    print("✅ ComfyUI 본체 확인됨.")

# 3. 모델 및 노드 경로 연결 (선택한 소스에 따라 스위칭)
print(f"🔄 {source_selection} 모드로 데이터 연결 중...")

# 기존 경로 삭제 (심볼릭 링크 꼬임 방지)
!rm -rf {LOCAL_PATH}/models
!rm -rf {LOCAL_PATH}/custom_nodes

if source_selection == "Google_Drive":
    # 구글 드라이브로 연결
    !ln -s {DRIVE_BASE}/models {LOCAL_PATH}/models
    !ln -s {DRIVE_BASE}/custom_nodes {LOCAL_PATH}/custom_nodes
    print(f"🔗 연결 완료: {DRIVE_BASE} <-> {LOCAL_PATH}")
else:
    # 로컬 안전 구역(/content/storage)으로 연결
    os.makedirs(f"{SAFE_STORAGE}/models", exist_ok=True)
    os.makedirs(f"{SAFE_STORAGE}/custom_nodes", exist_ok=True)
    !ln -s {SAFE_STORAGE}/models {LOCAL_PATH}/models
    !ln -s {SAFE_STORAGE}/custom_nodes {LOCAL_PATH}/custom_nodes
    print(f"🔗 연결 완료: {SAFE_STORAGE} <-> {LOCAL_PATH}")

# 4. 터널링 가동 (Cloudflared)
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

if os.path.exists("/content/tunnel.log"): os.remove("/content/tunnel.log")
subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
                 stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)

# 5. 접속 링크 출력
time.sleep(12)
try:
    with open("/content/tunnel.log", "r") as f:
        url = re.findall(r"https://[a-zA-Z0-9-.]+\.trycloudflare\.com", f.read())
        if url:
            print("\n" + "="*55 + f"\n🎉 접속 링크: {url[0]}\n" + "="*55 + "\n")
except: print("❌ 터널 링크 생성 실패")

# 6. 최종 실행 (메모리 최적화 옵션 포함)
if os.path.exists(LOCAL_PATH):
    %cd {LOCAL_PATH}
    # --disable-mmap: Qwen 2.5 같은 대용량 모델 로딩 시 RAM 부족 에러 방지
    # --cpu-vae: VAE 연산을 CPU로 돌려 GPU 메모리 확보
    # --lowvram: 저사양 모드 가동
    !python main.py --listen --preview-method auto --cpu-vae --lowvram --disable-mmap
else:
    print("❌ 치명적 오류: 실행 폴더를 찾을 수 없습니다.")


🎉 접속 링크: https://highly-thomas-consultancy-packing.trycloudflare.com

/content/storage/custom_nodes
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-03-28 18:08:20.170
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log

Prestartup times for custom nodes:
   6.0 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager

Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'scaled_mm_nvfp4']}
Found com